# GRU Model - Multiple Training Runs
This notebook trains the model N times and reports aggregate MAPE statistics

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import os
import sys
sys.path.append("..")


from utils import load_and_prepare_data, get_error_df
root = os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', 'Data')
denoised_sales_df = pd.read_excel(os.path.join(root, 'FixedData_Random_Smoothed.xlsx'))
noised_sales_df = pd.read_excel(os.path.join(root, 'DataMissingValuesFilled.xlsx'))
holiday_df = pd.read_csv(os.path.join(root, 'holidays.csv'))
crude_df = pd.read_csv(os.path.join(root, 'Crude Oil Prices.csv'))



val_split_date = '2023-06-01'
test_split_date = '2024-01-01'
seq_length = 30
forecast_length = 30

epochs = 150

shift_crude_days = 0
warmup_days = 14
warmup_epochs = 2
warmup_every_n_days = 1

act_fn = 'relu'
loss_fn = 'asymmetric_mse'


In [ ]:
from itertools import product
import json
N_RUNS = 20

with open(os.path.join(root, 'GRU Hyperparameters.txt'), "r") as f:
    hyperparameters = json.load(f)


EXPERIMENTS = [
    # -------------------- SALES ONLY --------------------
    {
        "name": "sales_only_scaled_no_calender",
        "use_scaling": True,
        "use_crude": False,
        "use_month": False,
        "use_month_sin_cos": False,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_only_scaled_calender_numbers",
        "use_scaling": True,
        "use_crude": False,
        "use_month": True,
        "use_month_sin_cos": False,
        "use_dayofweek": True,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_only_scaled_calender_sincos",
        "use_scaling": True,
        "use_crude": False,
        "use_month": False,
        "use_month_sin_cos": True,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": True,
    },

    # -------------------- SALES + CRUDE --------------------
    {
        "name": "sales_and_crude_scaled_no_calender",
        "use_scaling": True,
        "use_crude": True,
        "use_month": False,
        "use_month_sin_cos": False,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_and_crude_scaled_calender_numbers",
        "use_scaling": True,
        "use_crude": True,
        "use_month": True,
        "use_month_sin_cos": False,
        "use_dayofweek": True,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_and_crude_scaled_calender_sincos",
        "use_scaling": True,
        "use_crude": True,
        "use_month": False,
        "use_month_sin_cos": True,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": True,
    },
]


In [ ]:
def build_datasets_for_config(cfg):
  sales_scaler = MinMaxScaler() if cfg["use_scaling"] else None
  crude_scaler = MinMaxScaler() if cfg["use_scaling"] else None

  train_dataset, val_dataset, test_dataset = load_and_prepare_data(
      denoised_sales_df=denoised_sales_df,
      noised_sales_df=noised_sales_df,
      crude_df=crude_df,
      val_split_date=val_split_date,
      test_split_date=test_split_date,
      seq_length=seq_length,
      forecast_length=forecast_length,
      shift_crude_days=shift_crude_days,
      use_crude=cfg["use_crude"],
      use_month=cfg["use_month"],
      use_month_sin_cos=cfg["use_month_sin_cos"],
      use_dayofweek=cfg["use_dayofweek"],
      use_dayofweek_sin_cos=cfg["use_dayofweek_sin_cos"],
      sales_scaler=sales_scaler,
      crude_scaler=crude_scaler
  )

  return train_dataset, val_dataset, test_dataset


In [ ]:
from engine import train_model, generate_predictions
from holidayCorrection import holiday_correction
from Models import Transformer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

results = {}

for cfg in EXPERIMENTS:
    print(f"\n{'#'*70}")
    print(f"Experiment: {cfg['name']}")
    print(f"{'#'*70}\n")

    num_layers = hyperparameters[cfg['name']]['num_layers']
    d_model = hyperparameters[cfg['name']]['d_model']
    dim_feedforward = 4*d_model
    n_head = hyperparameters[cfg['name']]['n_head']
    dropout = hyperparameters[cfg['name']]['dropout']
    lr = hyperparameters[cfg['name']]['lr']
    batch_size = hyperparameters[cfg['name']]['batch_size']
    under_parameter = hyperparameters[cfg['name']]['under_parameter']
    over_parameter = hyperparameters[cfg['name']]['over_parameter']
    
    train_dataset, val_dataset, test_dataset = build_datasets_for_config(cfg)

    all_corrected_mapes = []
    all_over_prediction_errors = []
    all_under_prediction_errors = []

    for run_idx in range(N_RUNS):
        print(f"\n{'='*60}")
        print(f"Run {run_idx + 1}/{N_RUNS}")
        print(f"{'='*60}")

        input_dim = train_dataset.X.shape[2]

        model = Transformer(
            input_dim=input_dim,
            d_model=d_model,
            nhead=n_head,
            num_layers=num_layers,
            dim_feedforward=dim_feedforward,
            output_length=forecast_length,
            dropout=dropout
        ).to(device)

        if run_idx == 0:
            print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}\n")

        train_loader = train_dataset.get_dataloader(batch_size=batch_size, shuffle=True)
        val_loader = val_dataset.get_dataloader(batch_size=batch_size, shuffle=False)

        train_losses, test_losses, optimizer, criterion = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=epochs,
            lr=lr,
            device=device,
            under_parameter=under_parameter,
            over_parameter=over_parameter,
            patience=50,
            loss_fn=loss_fn
        )

        Y_pred_uncorrected, Y_true, Y_true_noised = generate_predictions(
            model, test_dataset, device=device
        )

        Y_pred_corrected = holiday_correction(Y_pred_uncorrected, test_dataset.sample_dates,noised_sales_df, holiday_df)

        error_df = get_error_df(
            Y_true_noised=Y_true_noised,
            Y_pred_corrected=Y_pred_corrected,
            Y_pred_uncorrected=Y_pred_uncorrected,
            sample_dates=test_dataset.sample_dates
        )

        corrected_mape = error_df['CorrectedMape'].mean()
        over_prediction_error = error_df['ErrorOverPrediction'].mean()
        under_prediction_error = error_df['ErrorUnderPrediction'].mean()

        all_corrected_mapes.append(corrected_mape)
        all_over_prediction_errors.append(over_prediction_error)
        all_under_prediction_errors.append(under_prediction_error)

        print(f"Run {run_idx+1} | MAPE: {corrected_mape:.3f}")

    results[cfg["name"]] = {
        "mape_mean": np.mean(all_corrected_mapes),
        "mape_std": np.std(all_corrected_mapes),
        "over_mean": np.mean(all_over_prediction_errors),
        "over_std":np.std(all_over_prediction_errors),
        "under_mean": np.mean(all_under_prediction_errors),
        "under_std": np.std(all_under_prediction_errors)

    }

    print(f"\n✔ Experiment {cfg['name']} completed")
    print(f"MAPE: {results[cfg['name']]['mape_mean']}")



Using device: cuda


######################################################################
Experiment: sales_only_no_scaled_no_calender
######################################################################

Total records : 1736
Train records : 1156
Val records   : 214
Test records  : 366

Val starts  at: 2023-06-01
Test starts at: 2024-01-01
Using features: ['Sales']

Run 1/20


/kaggle/input/datasets/milanroy/cdis-codebase-and-dataset/CodeBase and Dataset/CodeBase/utils.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'])


Model Parameters: 58,590

Epoch 10/150 | Train Loss: 1672934246.028986 | Val Loss: 706208317.333333
Epoch 20/150 | Train Loss: 1397914802.086957 | Val Loss: 940305862.666667
Epoch 30/150 | Train Loss: 1316800236.521739 | Val Loss: 571556230.666667
Epoch 40/150 | Train Loss: 1347602861.449275 | Val Loss: 598500960.000000
Epoch 50/150 | Train Loss: 1276629577.275362 | Val Loss: 889803201.333333
Epoch 60/150 | Train Loss: 1275831914.666667 | Val Loss: 836543636.000000
Epoch 70/150 | Train Loss: 1199517471.536232 | Val Loss: 691194385.333333
Epoch 80/150 | Train Loss: 1178882278.028986 | Val Loss: 581452932.000000
Epoch 90/150 | Train Loss: 1212766937.043478 | Val Loss: 648232645.333333
Epoch 100/150 | Train Loss: 1148718608.695652 | Val Loss: 577259893.333333
Epoch 110/150 | Train Loss: 1150861329.623188 | Val Loss: 581797056.000000
Epoch 120/150 | Train Loss: 1145054156.985507 | Val Loss: 551811273.333333
Epoch 130/150 | Train Loss: 1212898856.811594 | Val Loss: 585865644.000000
Epoch 14

/kaggle/input/datasets/milanroy/cdis-codebase-and-dataset/CodeBase and Dataset/CodeBase/utils.py:261: RuntimeWarning: Mean of empty slice
  under_mean = np.nanmean(under_errors, axis=1)
/kaggle/input/datasets/milanroy/cdis-codebase-and-dataset/CodeBase and Dataset/CodeBase/utils.py:262: RuntimeWarning: Mean of empty slice
  over_mean  = np.nanmean(over_errors, axis=1)


Run 1 | MAPE: 7.165

Run 2/20
Epoch 10/150 | Train Loss: 1618733211.826087 | Val Loss: 684340779.333333
Epoch 20/150 | Train Loss: 1343027069.217391 | Val Loss: 616225959.333333
Epoch 30/150 | Train Loss: 1234431067.826087 | Val Loss: 577903969.333333
Epoch 40/150 | Train Loss: 1223300662.724638 | Val Loss: 574874060.000000
Epoch 50/150 | Train Loss: 1229845221.101449 | Val Loss: 635442836.000000
Epoch 60/150 | Train Loss: 1183364923.362319 | Val Loss: 625363181.333333
Epoch 70/150 | Train Loss: 1237877057.855072 | Val Loss: 741130357.333333
Epoch 80/150 | Train Loss: 1212335831.188406 | Val Loss: 611050292.000000
Epoch 90/150 | Train Loss: 1231414321.159420 | Val Loss: 789985302.666667
Epoch 100/150 | Train Loss: 1120766159.768116 | Val Loss: 559202560.000000
Epoch 110/150 | Train Loss: 1110968299.594203 | Val Loss: 604891786.666667
Epoch 120/150 | Train Loss: 1131985382.956522 | Val Loss: 895680988.000000
Epoch 130/150 | Train Loss: 1140817369.971014 | Val Loss: 579643116.000000

Ear

...

/kaggle/input/datasets/milanroy/cdis-codebase-and-dataset/CodeBase and Dataset/CodeBase/utils.py:261: RuntimeWarning: Mean of empty slice
  under_mean = np.nanmean(under_errors, axis=1)
/kaggle/input/datasets/milanroy/cdis-codebase-and-dataset/CodeBase and Dataset/CodeBase/utils.py:262: RuntimeWarning: Mean of empty slice
  over_mean  = np.nanmean(over_errors, axis=1)


Run 18 | MAPE: 8.537

Run 19/20
Epoch 10/150 | Train Loss: 0.004612 | Val Loss: 0.003390
Epoch 20/150 | Train Loss: 0.003498 | Val Loss: 0.003773
Epoch 30/150 | Train Loss: 0.003077 | Val Loss: 0.002068
Epoch 40/150 | Train Loss: 0.002836 | Val Loss: 0.001914
Epoch 50/150 | Train Loss: 0.003035 | Val Loss: 0.002197
Epoch 60/150 | Train Loss: 0.002748 | Val Loss: 0.003815
Epoch 70/150 | Train Loss: 0.002601 | Val Loss: 0.002861

Early stopping triggered at epoch 71. Best Val Loss: 0.001938


/kaggle/input/datasets/milanroy/cdis-codebase-and-dataset/CodeBase and Dataset/CodeBase/utils.py:261: RuntimeWarning: Mean of empty slice
  under_mean = np.nanmean(under_errors, axis=1)
/kaggle/input/datasets/milanroy/cdis-codebase-and-dataset/CodeBase and Dataset/CodeBase/utils.py:262: RuntimeWarning: Mean of empty slice
  over_mean  = np.nanmean(over_errors, axis=1)


Run 19 | MAPE: 7.843

Run 20/20
Epoch 10/150 | Train Loss: 0.004235 | Val Loss: 0.002361
Epoch 20/150 | Train Loss: 0.003393 | Val Loss: 0.001971
Epoch 30/150 | Train Loss: 0.003118 | Val Loss: 0.002467
Epoch 40/150 | Train Loss: 0.002945 | Val Loss: 0.002749
Epoch 50/150 | Train Loss: 0.002801 | Val Loss: 0.002505
Epoch 60/150 | Train Loss: 0.002697 | Val Loss: 0.002283
Epoch 70/150 | Train Loss: 0.002691 | Val Loss: 0.001844
Epoch 80/150 | Train Loss: 0.002619 | Val Loss: 0.002738

Early stopping triggered at epoch 89. Best Val Loss: 0.001806
Run 20 | MAPE: 7.803

✔ Experiment sales_and_crude_scaled_calender_sincos completed
MAPE: 8.48408559113014


/kaggle/input/datasets/milanroy/cdis-codebase-and-dataset/CodeBase and Dataset/CodeBase/utils.py:261: RuntimeWarning: Mean of empty slice
  under_mean = np.nanmean(under_errors, axis=1)
/kaggle/input/datasets/milanroy/cdis-codebase-and-dataset/CodeBase and Dataset/CodeBase/utils.py:262: RuntimeWarning: Mean of empty slice
  over_mean  = np.nanmean(over_errors, axis=1)


In [6]:
# Calculate and display aggregate statistics
print(f"\n\n{'='*60}")
print(f"AGGREGATE RESULTS ACROSS {N_RUNS} RUNS")
print(f"{'='*60}\n")

for cfg in EXPERIMENTS:
  print(f"{cfg['name']}")
  print(f"MAPE: {results[cfg['name']]['mape_mean']:.3f} ± {results[cfg['name']]['mape_std']:.3f}")
  print(f"Over Predcition Error: {results[cfg['name']]['over_mean']:.3f} ± {results[cfg['name']]['over_std']:.3f}")
  print(f"Under Predcition Error: {results[cfg['name']]['under_mean']:.3f} ± {results[cfg['name']]['under_std']:.3f}")
  print()



AGGREGATE RESULTS ACROSS 20 RUNS

sales_only_no_scaled_no_calender
MAPE: 7.185 ± 0.048
Over Predcition Error: 8.218 ± 0.105
Under Predcition Error: -4.650 ± 0.139

sales_only_no_scaled_calender_numbers
MAPE: 7.042 ± 0.060
Over Predcition Error: 8.095 ± 0.141
Under Predcition Error: -4.742 ± 0.188

sales_only_no_scaled_calender_sincos
MAPE: 7.116 ± 0.055
Over Predcition Error: 8.104 ± 0.120
Under Predcition Error: -5.000 ± 0.216

sales_only_scaled_no_calender
MAPE: 7.226 ± 0.106
Over Predcition Error: 8.343 ± 0.211
Under Predcition Error: -4.801 ± 0.330

sales_only_scaled_calender_numbers
MAPE: 7.125 ± 0.130
Over Predcition Error: 8.180 ± 0.184
Under Predcition Error: -4.957 ± 0.322

sales_only_scaled_calender_sincos
MAPE: 8.421 ± 0.510
Over Predcition Error: 9.485 ± 0.338
Under Predcition Error: -5.927 ± 0.830

sales_and_crude_no_scaled_no_calender
MAPE: 7.123 ± 0.055
Over Predcition Error: 8.042 ± 0.119
Under Predcition Error: -5.007 ± 0.208

sales_and_crude_no_scaled_calender_numbe

In [7]:
import json

with open('Results.txt', 'w') as f:
    json.dump(results, f, indent=4)
